In [2]:
import os
import pydicom
import numpy as np
import cv2

In [3]:
input_directory = os.path.join("..", "data", "PANCREAS_2", "PANCREAS_2")
output_directory = os.path.join("..", "data", "PANCREAS_PREPROCESSED")
# Create the output folder if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

In [4]:
color_threshold = 200
processed_count = 0

print("Starting Batch Segmentation...")
print(f"Reading from: {os.path.abspath(input_directory)}")
print(f"Saving to:    {os.path.abspath(output_directory)}\n")

# BATCH PROCESSING LOOP
for study_id in os.listdir(input_directory):
    # Skip hidden files/folders
    if study_id.startswith("."): 
        continue
        
    study_path = os.path.join(input_directory, study_id)
    
    # Go into the date folder
    date_folders = [f for f in os.listdir(study_path) if not f.startswith(".")]
    if not date_folders:
        continue
    date_path = os.path.join(study_path, date_folders[0])
    
    # Get the single DICOM file
    files = [f for f in os.listdir(date_path) if not f.startswith(".")]
    if not files:
        continue
    dicom_path = os.path.join(date_path, files[0])
    
    try:
        # 1. Load the Image
        dataset = pydicom.dcmread(dicom_path)
        image_rgb = dataset.pixel_array
        
        # 2. Isolate the White Pixels
        white_mask = (image_rgb[:, :, 0] > color_threshold) & \
                     (image_rgb[:, :, 1] > color_threshold) & \
                     (image_rgb[:, :, 2] > color_threshold)
        white_mask_8bit = (white_mask * 255).astype(np.uint8)
        
        # 3. Clean the Noise (Keep Largest Island)
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(white_mask_8bit, connectivity=8)
        
        largest_label = 1
        max_area = 0
        for i in range(1, num_labels):
            area = stats[i, cv2.CC_STAT_AREA]
            if area > max_area:
                max_area = area
                largest_label = i
                
        clean_contour = np.zeros_like(white_mask_8bit)
        if max_area > 0:
            clean_contour[labels == largest_label] = 255
            
        # Bridge gaps in the contour using morphological closing
        
        kernel = np.ones((20, 20), np.uint8)
        closed_contour = cv2.morphologyEx(clean_contour, cv2.MORPH_CLOSE, kernel)
            
        # 4. Fill the Contour (The Paint Bucket)
        # findContours finds the mathematical boundary of our closed white loop
        contours, hierarchy = cv2.findContours(closed_contour, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Create a blank black mask, and draw the contour filled in with white (-1 / FILLED)
        filled_mask = np.zeros_like(closed_contour)
        cv2.drawContours(filled_mask, contours, -1, color=255, thickness=cv2.FILLED)
        
        # 5. Convert original to Grayscale (PyRadiomics prefers grayscale intensity)
        gray_image = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
        
        # 6. Save the pair of files (Image + Mask)
        # We use standard .png format, which is lightweight and PyRadiomics can read natively
        image_save_path = os.path.join(output_directory, f"{study_id}_image.png")
        mask_save_path = os.path.join(output_directory, f"{study_id}_mask.png")
        
        cv2.imwrite(image_save_path, gray_image)
        cv2.imwrite(mask_save_path, filled_mask)
        
        processed_count += 1
        
        # Print progress every 20 images
        if processed_count % 20 == 0:
            print(f"Processed {processed_count} images...")
            
    except Exception as e:
        print(f"Error processing {study_id}: {e}")

print(f"\nTotal processed: {processed_count}")
print(f"Check the folder: {os.path.abspath(output_directory)}")

Starting Batch Segmentation...
Reading from: /home/daniduhnev/projects/master-thesis/data/PANCREAS_2/PANCREAS_2
Saving to:    /home/daniduhnev/projects/master-thesis/data/PANCREAS_PREPROCESSED

Processed 20 images...
Processed 40 images...
Processed 60 images...
Processed 80 images...
Processed 100 images...
Processed 120 images...

Total processed: 134
Check the folder: /home/daniduhnev/projects/master-thesis/data/PANCREAS_PREPROCESSED
